# 8. Fetching Data from NCBI, UniProt and Ensembl

The programs of this chapter, in the order the book prints
them. Each cell holds the file of the same name in this folder,
so editing a cell does not change the script beside it.

Run the cells from this folder, where the data files the chapter
uses live. Some cells call command line tools or download data,
and a few take minutes.


## `lookup_hbb.py`

Ask Ensembl's REST API about one gene and print the JSON answer.


In [ ]:
# Ask Ensembl's REST API about one gene and print the JSON answer.
import json
import urllib.request

URL = "https://rest.ensembl.org/lookup/symbol/homo_sapiens/HBB"
request = urllib.request.Request(
    URL, headers={"Accept": "application/json"}
)

with urllib.request.urlopen(request) as response:
    answer = json.load(response)
print(json.dumps(answer, indent=2))

## `ncbi_hbb.py`

Search NCBI for human HBB records and download one, with Biopython's Entrez module.


In [ ]:
# Search NCBI for human HBB records and download one, with Biopython's
# Entrez module.
import os
from pathlib import Path

from Bio import Entrez, SeqIO

Entrez.email = "your.name@example.org"  # NCBI asks for a real address
Entrez.api_key = os.environ.get("NCBI_API_KEY")  # optional, see text
Entrez.tool = "bioinformatics-python-book"

query = "HBB[Gene Name] AND Homo sapiens[Organism] AND refseq[filter]"
with Entrez.esearch(db="nuccore", term=query) as handle:
    found = Entrez.read(handle)
print(f"{found['Count']} RefSeq records match the search")

with Entrez.esummary(
    db="nuccore", id=",".join(found["IdList"])
) as handle:
    summaries = Entrez.read(handle)
for summary in summaries:
    print(f"  {summary['AccessionVersion']:15} {summary['Title'][:52]}")

cache = Path("NM_000518.5.gb")
if not cache.exists():
    with Entrez.efetch(
        db="nuccore", id="NM_000518.5", rettype="gb", retmode="text"
    ) as handle:
        cache.write_text(handle.read())
record = SeqIO.read(cache, "genbank")
print(f"\n{record.id}: {len(record)} bp, {record.description}")

## `globins.py`

Download a set of globin proteins from UniProt, once, as FASTA.


In [ ]:
# Download a set of globin proteins from UniProt, once, as FASTA.
import urllib.request
from pathlib import Path

from Bio import SeqIO

# Beta globins from twelve vertebrates, then alpha globins and myoglobin
ACCESSIONS = [
    "P68871",
    "P68873",
    "P02088",
    "P02091",
    "P02070",
    "P02067",
    "P02062",
    "P60524",
    "P02111",
    "P02112",
    "P02132",
    "Q90486",
    "P69905",
    "P01942",
    "P01994",
    "P02144",
]
UNIPROT = "https://rest.uniprot.org/uniprotkb/accessions"
URL = UNIPROT + "?format=fasta&accessions="

target = Path("globins.fasta")
if target.exists():
    print(f"using the copy already in {target}")
else:
    with urllib.request.urlopen(URL + ",".join(ACCESSIONS)) as response:
        target.write_bytes(response.read())
    print(f"downloaded {target}")

for record in SeqIO.parse(target, "fasta"):
    entry = record.id.split("|")[2]
    species = record.description.split("OS=")[1].split(" OX=")[0]
    print(f"  {entry:11} {len(record):4} aa  {species}")

## `identifiers.py`

The human globin genes in Ensembl, joined to NCBI's gene table from Chapter 5 on their shared NCBI GeneID.


In [ ]:
# The human globin genes in Ensembl, joined to NCBI's gene table from
# Chapter 5 on their shared NCBI GeneID.
import json
import time
import urllib.request

import pandas as pd

ENSEMBL = "https://rest.ensembl.org"
GENES = ["HBB", "HBD", "HBG1", "HBG2", "HBE1", "HBA1", "HBA2", "HBZ"]


def get_json(path: str) -> dict | list:
    request = urllib.request.Request(
        ENSEMBL + path, headers={"Content-Type": "application/json"}
    )
    with urllib.request.urlopen(request) as response:
        return json.load(response)


rows = []
for symbol in GENES:
    gene = get_json(f"/lookup/symbol/homo_sapiens/{symbol}")
    xrefs = get_json(f"/xrefs/id/{gene['id']}?all_levels=1")
    by_database = {x["dbname"]: x["primary_id"] for x in xrefs}
    rows.append(
        {
            "symbol": symbol,
            "ensembl": gene["id"],
            "GeneID": int(by_database["EntrezGene"]),
            "uniprot": by_database.get("Uniprot/SWISSPROT"),
            "ens_start": gene["start"],
            "ens_end": gene["end"],
        }
    )
    time.sleep(0.1)  # Ensembl allows 15 requests a second

ensembl = pd.DataFrame(rows)
ncbi = pd.read_csv(
    "../ch05/genes.tsv", sep="\t", dtype={"chromosome": str}
)
table = ensembl.merge(
    ncbi[["GeneID", "chromosome", "start", "end"]], on="GeneID"
)
table = table.rename(columns={"start": "ncbi_start", "end": "ncbi_end"})
print(table[["symbol", "ensembl", "GeneID", "uniprot"]].to_string())
print()
print(
    table[
        [
            "symbol",
            "chromosome",
            "ncbi_start",
            "ens_start",
            "ncbi_end",
            "ens_end",
        ]
    ].to_string()
)